In [0]:
file_path = "/Volumes/workspace/default/cs5224_lab02/twitter_combined.txt"

raw_df = spark.read.text(file_path)

raw_df.show(10, truncate=False)

print("Total raw records:", raw_df.count())

+-------------------+
|value              |
+-------------------+
|214328887 34428380 |
|17116707 28465635  |
|380580781 18996905 |
|221036078 153460275|
|107830991 17868918 |
|151338729 222261763|
|19705747 34428380  |
|222261763 88323281 |
|19933035 149538028 |
|158419434 17434613 |
+-------------------+
only showing top 10 rows
Total raw records: 2426143


In [0]:
from pyspark.sql import functions as F

parsed_df = raw_df.select(
    F.split(F.trim(F.col("value")), r"\s+").alias("parts")
)

valid_df = (
    parsed_df
    .filter(F.size("parts") == 2)
    .filter(
        F.col("parts")[0].rlike(r"^[0-9]+$") &
        F.col("parts")[1].rlike(r"^[0-9]+$")
    )
    .select(
        F.col("parts")[0].alias("follower"),
        F.col("parts")[1].alias("followee")
    )
)

clean_df = valid_df.dropDuplicates(["follower", "followee"])

print("Raw records:", raw_df.count())
print("Valid records:", valid_df.count())
print("Unique follow relationships:", clean_df.count())
print("Invalid records:", raw_df.count() - valid_df.count())
print("Duplicate records:", valid_df.count() - clean_df.count())

clean_df.show(10, truncate=False)


Raw records: 2426143
Valid records: 2426143
Unique follow relationships: 1768149
Invalid records: 0
Duplicate records: 657994
+---------+---------+
|follower |followee |
+---------+---------+
|280935165|437804658|
|204317520|160237722|
|324201646|40981798 |
|43003845 |314316607|
|206923844|31331740 |
|103598216|270449528|
|204317520|239144776|
|406628822|56860418 |
|265077741|40981798 |
|116498875|26929220 |
+---------+---------+
only showing top 10 rows


In [0]:
from pyspark.sql import functions as F

followers_df = (
    clean_df
    .groupBy("followee")
    .count()
    .withColumnRenamed("followee", "user_id")
    .withColumnRenamed("count", "followers_count"))

followees_df = (
    clean_df
    .groupBy("follower")
    .count()
    .withColumnRenamed("follower", "user_id")
    .withColumnRenamed("count", "followees_count"))

all_users_df = (
    clean_df.select(F.col("follower").alias("user_id"))
    .union(clean_df.select(F.col("followee").alias("user_id")))
    .distinct())

result_df = (
    all_users_df
    .join(followers_df, on="user_id", how="left")
    .join(followees_df, on="user_id", how="left")
    .fillna(0, subset=["followers_count", "followees_count"]))

result_df.filter(
    F.col("user_id").isin("214328887", "107830991")
).orderBy("user_id").show(truncate=False)

print("Total unique users:", result_df.count())

+---------+---------------+---------------+
|user_id  |followers_count|followees_count|
+---------+---------------+---------------+
|107830991|180            |157            |
|214328887|180            |214            |
+---------+---------------+---------------+

Total unique users: 81306


In [0]:
table_name = "workspace.default.twitter_user_stats"

(
    result_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(table_name))

print("Delta table created successfully:", table_name)

saved_df = spark.table(table_name)

saved_df.filter(
    F.col("user_id").isin("214328887", "107830991")
).orderBy("user_id").show(truncate=False)

print("Total saved users:", saved_df.count())

Delta table created successfully: workspace.default.twitter_user_stats
+---------+---------------+---------------+
|user_id  |followers_count|followees_count|
+---------+---------------+---------------+
|107830991|180            |157            |
|214328887|180            |214            |
+---------+---------------+---------------+

Total saved users: 81306
